# 01 — REFIT data exploration

Thin notebook. All logic lives in `src/`. Run `scripts/build_panel.py` first if the processed panel is missing.

Required figures land in `reports/figures/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if not (ROOT / "VERSION").exists():
    ROOT = Path("/kaggle/working/household-energy-forecasting")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_data_config, load_weather_config
from src.data.loading import discover_house_files
from src.data.metadata import household_table
from src.data.pipeline import build_hourly_panel, load_processed_panel
from src.eval.plots import (
    plot_corr_heatmap,
    plot_daily_profile,
    plot_load_vs_temperature,
    plot_missing_heatmap,
)
from src.graphs.correlation import pearson_knn
from src.paths import figures_dir, processed_dir, resolve_raw_dir

data_cfg = load_data_config()
weather_cfg = load_weather_config()
raw_dir = resolve_raw_dir(data_cfg)
print("repo", ROOT)
print("raw_dir", raw_dir)
print("discovered", sorted(discover_house_files(raw_dir)))

In [ ]:
panel_path = processed_dir(data_cfg) / "hourly_panel.parquet"
if panel_path.exists():
    panel, split = load_processed_panel(data_cfg)
else:
    bundle = build_hourly_panel(data_cfg, weather_cfg)
    panel, split = bundle.panel, bundle.split

print(panel.head())
print("kept", sorted(panel["household_id"].unique()))
print(split)
print("observed share", float(panel["observed"].mean()))

In [ ]:
summary = (
    panel.groupby("household_id")
    .agg(
        n_hours=("timestamp", "count"),
        n_observed=("observed", "sum"),
        coverage=("observed", "mean"),
        first=("timestamp", "min"),
        last=("timestamp", "max"),
        mean_kwh=("kwh", "mean"),
        median_kwh=("kwh", "median"),
        max_kwh=("kwh", "max"),
    )
    .reset_index()
)
meta = household_table()
summary = summary.merge(meta, on="household_id", how="left")
summary

In [ ]:
fig = figures_dir(data_cfg)
kept = sorted(int(x) for x in panel["household_id"].unique())
plot_missing_heatmap(panel, fig / "missing_heatmap.png")
plot_daily_profile(panel, fig / "daily_profile.png")
adj, corr = pearson_knn(panel, kept, k=3)
plot_corr_heatmap(corr, fig / "train_pearson.png")
plot_load_vs_temperature(panel, fig / "load_vs_temperature.png")
print("figures", list(fig.glob("*.png")))
print("graph shape", adj.shape)

Open `reports/data_quality.md` after this notebook. That file is what Chapter 3 should cite for counts, not the 2017 paper.